In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
import torchvision.models as models
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import os
import random
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
from collections import Counter
from typing import Tuple, Dict, List
import time
from tqdm import tqdm
import copy

import torch # Đảm bảo đã import
from torch.utils.data import Dataset
from torchvision import transforms

In [4]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from tqdm import tqdm
from PIL import Image
import cv2
from pathlib import Path
import numpy as np
# Define emotion mapping
EMOTION_MAPPING = {
    1: 'surprise',
    2: 'fear',
    3: 'disgust',
    4: 'happiness',
    5: 'sadness',
    6: 'anger',
    7: 'neutral'
}



# Configuration
DATA_PATH = '/kaggle/working/Heatmaps_RafDb_MP_DLIB_sigma_15' 
BATCH_SIZE = 128
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TARGET_SIZE = (112, 112)  # Size for landmark detection
NUM_EPOCHS = 120
MODEL_SAVE_PATH = '/kaggle/working/best_model.pth'
class FERPlusDataset(Dataset):
    """
    Dataset class for FERPlus dataset with pre-divided train/val/test splits
    """
    def __init__(self, root_dir, split='train', image_size=(112, 112), transform_type=None):
        self.root_dir = Path(root_dir)
        self.split = split
        self.image_size = image_size
        self.transform_type = transform_type
        
        self.global_mean = [0.485, 0.456, 0.406]
        self.global_std = [0.229, 0.224, 0.225]

        
        # Prepare dataset
        self.images, self.labels, self.heat_images = self._load_dataset()
        
        # Print dataset statistics
        self._print_statistics()
        
        # Prepare transforms
        self.base_transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(self.image_size),
            transforms.ToTensor(),
        ])
        self.image_transform = transforms.Compose([
                self.base_transform,
                transforms.Normalize(mean=self.global_mean, std=self.global_std)
        ])
        
        
        # Additional train transforms
        self.train_transform = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(10)
        ])
    
    def _load_dataset(self):
        """Load dataset from preprocessed .npz file"""
        saved_data_path = os.path.join(self.root_dir, f'{self.split}.npz')
        
        try:
            data = np.load(saved_data_path)
            return data['images'], data['labels'], data['heatmaps']
        except Exception as e:
            print(f"Error loading dataset: {e}")
            return [], [], []
    
    def _print_statistics(self):
        """Print dataset statistics"""
        unique_labels, counts = np.unique(self.labels, return_counts=True)
        total_images = len(self.labels)
        
        print(f"\n{self.split} set statistics:")
        print(f"Total images: {total_images}")
        
        for label, count in zip(unique_labels, counts):
            emotion = EMOTION_MAPPING[label]
            percentage = (count / total_images) * 100
            print(f"{emotion}: {count} images ({percentage:.2f}%)")
    

    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        # Load image, label, and heatmap
        image = self.images[idx]
        # print(" ảnh gốc:", image.shape)  #  100x100x3
        label = self.labels[idx] - 1
        
        # Transform image
        transformed_image = self.image_transform(image)
        # print("ảnh sau biến đổi:", transformed_image.shape)  # 3x224x224
        
        if self.transform_type == 'train':
            transformed_image = self.train_transform(transformed_image)
    
        
        return transformed_image, label

In [5]:
train_dataset = FERPlusDataset(DATA_PATH, split='train_split', transform_type='train')
val_dataset = FERPlusDataset(DATA_PATH, split='val')
test_dataset = FERPlusDataset(DATA_PATH, split='test')


train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, num_workers=4)


train_split set statistics:
Total images: 8540
surprise: 937 images (10.97%)
fear: 190 images (2.22%)
disgust: 511 images (5.98%)
happiness: 3366 images (39.41%)
sadness: 1390 images (16.28%)
anger: 437 images (5.12%)
neutral: 1709 images (20.01%)

val set statistics:
Total images: 2142
surprise: 235 images (10.97%)
fear: 49 images (2.29%)
disgust: 130 images (6.07%)
happiness: 844 images (39.40%)
sadness: 341 images (15.92%)
anger: 112 images (5.23%)
neutral: 431 images (20.12%)

test set statistics:
Total images: 2676
surprise: 303 images (11.32%)
fear: 63 images (2.35%)
disgust: 141 images (5.27%)
happiness: 1054 images (39.39%)
sadness: 415 images (15.51%)
anger: 131 images (4.90%)
neutral: 569 images (21.26%)


In [6]:
from torch.nn import Linear, Conv2d, BatchNorm1d, BatchNorm2d, PReLU, ReLU, Sigmoid, Dropout2d, Dropout, AvgPool2d, \
    MaxPool2d, AdaptiveAvgPool2d, Sequential, Module, Parameter
import torch.nn.functional as F
import torch
import torch.nn as nn
from collections import namedtuple
import math
import pdb


##################################  Original Arcface Model #############################################################
######## ccc#######################
class Flatten(Module):
    def forward(self, input):
        return input.view(input.size(0), -1)


##################################  MobileFaceNet #############################################################

class Conv_block(Module):
    def __init__(self, in_c, out_c, kernel=(1, 1), stride=(1, 1), padding=(0, 0), groups=1):
        super(Conv_block, self).__init__()
        self.conv = Conv2d(in_c, out_channels=out_c, kernel_size=kernel, groups=groups, stride=stride, padding=padding,
                           bias=False)
        self.bn = BatchNorm2d(out_c)
        self.prelu = PReLU(out_c)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.prelu(x)
        return x


class Linear_block(Module):
    def __init__(self, in_c, out_c, kernel=(1, 1), stride=(1, 1), padding=(0, 0), groups=1):
        super(Linear_block, self).__init__()
        self.conv = Conv2d(in_c, out_channels=out_c, kernel_size=kernel, groups=groups, stride=stride, padding=padding,
                           bias=False)
        self.bn = BatchNorm2d(out_c)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        return x


class Depth_Wise(Module):
    def __init__(self, in_c, out_c, residual=False, kernel=(3, 3), stride=(2, 2), padding=(1, 1), groups=1):
        super(Depth_Wise, self).__init__()
        self.conv = Conv_block(in_c, out_c=groups, kernel=(1, 1), padding=(0, 0), stride=(1, 1))
        self.conv_dw = Conv_block(groups, groups, groups=groups, kernel=kernel, padding=padding, stride=stride)
        self.project = Linear_block(groups, out_c, kernel=(1, 1), padding=(0, 0), stride=(1, 1))
        self.residual = residual

    def forward(self, x):
        if self.residual:
            short_cut = x
        x = self.conv(x)
        x = self.conv_dw(x)
        x = self.project(x)
        if self.residual:
            output = short_cut + x
        else:
            output = x
        return output


class Residual(Module):
    def __init__(self, c, num_block, groups, kernel=(3, 3), stride=(1, 1), padding=(1, 1)):
        super(Residual, self).__init__()
        modules = []
        for _ in range(num_block):
            modules.append(
                Depth_Wise(c, c, residual=True, kernel=kernel, padding=padding, stride=stride, groups=groups))
        self.model = Sequential(*modules)

    def forward(self, x):
        return self.model(x)


class GNAP(Module):
    def __init__(self, embedding_size):
        super(GNAP, self).__init__()
        assert embedding_size == 512
        self.bn1 = BatchNorm2d(512, affine=False)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.bn2 = BatchNorm1d(512, affine=False)

    def forward(self, x):
        x = self.bn1(x)
        x_norm = torch.norm(x, 2, 1, True)
        x_norm_mean = torch.mean(x_norm)
        weight = x_norm_mean / x_norm
        x = x * weight
        x = self.pool(x)
        x = x.view(x.shape[0], -1)
        feature = self.bn2(x)
        return feature


class GDC(Module):
    def __init__(self, embedding_size):
        super(GDC, self).__init__()
        self.conv_6_dw = Linear_block(512, 512, groups=512, kernel=(7, 7), stride=(1, 1), padding=(0, 0))
        self.conv_6_flatten = Flatten()
        self.linear = Linear(512, embedding_size, bias=False)
        # self.bn = BatchNorm1d(embedding_size, affine=False)
        self.bn = BatchNorm1d(embedding_size)

    def forward(self, x):
        x = self.conv_6_dw(x)    #### [B, 512, 1, 1]
        x = self.conv_6_flatten(x)   #### [B, 512]
        x = self.linear(x)      #### [B, 136]
        x = self.bn(x)
        return x


class MobileFaceNet(Module):
    def __init__(self, input_size, embedding_size=512, output_name="GDC"):
        super(MobileFaceNet, self).__init__()
        assert output_name in ["GNAP", 'GDC']
        assert input_size[0] in [112]
        self.conv1 = Conv_block(3, 64, kernel=(3, 3), stride=(2, 2), padding=(1, 1))
        self.conv2_dw = Conv_block(64, 64, kernel=(3, 3), stride=(1, 1), padding=(1, 1), groups=64)
        self.conv_23 = Depth_Wise(64, 64, kernel=(3, 3), stride=(2, 2), padding=(1, 1), groups=128)
        self.conv_3 = Residual(64, num_block=4, groups=128, kernel=(3, 3), stride=(1, 1), padding=(1, 1))
        self.conv_34 = Depth_Wise(64, 128, kernel=(3, 3), stride=(2, 2), padding=(1, 1), groups=256)
        self.conv_4 = Residual(128, num_block=6, groups=256, kernel=(3, 3), stride=(1, 1), padding=(1, 1))
        self.conv_45 = Depth_Wise(128, 128, kernel=(3, 3), stride=(2, 2), padding=(1, 1), groups=512)
        self.conv_5 = Residual(128, num_block=2, groups=256, kernel=(3, 3), stride=(1, 1), padding=(1, 1))
        self.conv_6_sep = Conv_block(128, 512, kernel=(1, 1), stride=(1, 1), padding=(0, 0))
        if output_name == "GNAP":
            self.output_layer = GNAP(512)
        else:
            self.output_layer = GDC(embedding_size)

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    m.bias.data.zero_()
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    m.bias.data.zero_()

    def forward(self, x):
        out = self.conv1(x)
        # print(out.shape)
        out = self.conv2_dw(out)
        # print(out.shape)
        out = self.conv_23(out)
        # print(out.shape)
        out3 = self.conv_3(out)
        # print(out.shape)
        out = self.conv_34(out3)
        # print(out.shape)
        out4 = self.conv_4(out)  # [128, 14, 14]
        # print(out.shape)
        out = self.conv_45(out4)  # [128, 7, 7]
        # print(out.shape)
        out = self.conv_5(out)  # [128, 7, 7]
        # print(out.shape)
        conv_features = self.conv_6_sep(out)    ##### [B, 512, 7, 7]
        out = self.output_layer(conv_features)  ##### [B, 136]
        return out
mobilefacenet = MobileFaceNet(input_size=[112, 112], embedding_size=136)
print(mobilefacenet)

MobileFaceNet(
  (conv1): Conv_block(
    (conv): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (prelu): PReLU(num_parameters=64)
  )
  (conv2_dw): Conv_block(
    (conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64, bias=False)
    (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (prelu): PReLU(num_parameters=64)
  )
  (conv_23): Depth_Wise(
    (conv): Conv_block(
      (conv): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (prelu): PReLU(num_parameters=128)
    )
    (conv_dw): Conv_block(
      (conv): Conv2d(128, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=128, bias=False)
      (bn): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_st

In [7]:
# loss_plot_path = "/kaggle/working/loss_plot.png"
# accuracy_plot_path = "/kaggle/working/accuracy_plot.png"
def train_model(model, train_loader, val_loader, device, num_epochs=50, model_save_path='models/fer_model.pth'):
    """Train the model"""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Learning rate scheduler
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.1)
    # scaler = GradScaler()

    best_val_loss = float('inf')
    best_val_acc = 0.0
    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')
        for inputs, labels  in pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            pbar.set_postfix({'loss': running_loss/len(train_loader), 'acc': 100.*correct/total})

        train_loss = running_loss/len(train_loader)
        train_acc = 100.*correct/total
        train_losses.append(train_loss)
        train_accs.append(train_acc)

        # Validation phase
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        val_loss = val_loss/len(val_loader)
        val_acc = 100.*correct/total
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        print(f'Epoch {epoch+1}/{num_epochs}:')
        print(f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%')
        print(f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')

        # Learning rate scheduling
        scheduler.step(val_loss)

        # Save best model based on validation accuracy
        if val_acc > best_val_acc:
            print('The model which has the best validation accuracy is saved...')
            best_val_acc = val_acc
            print(f'Best Epoch {epoch+1}/{num_epochs}:')
            print("Best Validation Accuracy: ", best_val_acc)
            best_val_acc = val_acc
            torch.save(model.state_dict(), model_save_path)

    # Plot training history
    plot_training_history(train_losses, val_losses, train_accs, val_accs)

    return model
import matplotlib.pyplot as plt

def plot_training_history(train_losses, val_losses, train_accs, val_accs):
    """Plot training and validation metrics and save figures separately."""

    # Define save paths
    loss_plot_path = "/kaggle/working/loss_plot.png"
    accuracy_plot_path = "/kaggle/working/accuracy_plot.png"

    # Plot and save loss separately
    plt.figure(figsize=(8, 5))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Val Loss')
    plt.title('Model Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig(loss_plot_path, dpi=300, bbox_inches='tight')
    plt.close()  # Close figure to prevent overlap

    # Plot and save accuracy separately
    plt.figure(figsize=(8, 5))
    plt.plot(train_accs, label='Train Acc')
    plt.plot(val_accs, label='Val Acc')
    plt.title('Model Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.savefig(accuracy_plot_path, dpi=300, bbox_inches='tight')
    plt.close()  # Close figure to prevent overlap

    # Show both plots
    plt.show()

    print(f"Loss plot saved at: {loss_plot_path}")
    print(f"Accuracy plot saved at: {accuracy_plot_path}")

def count_parameters(model):
    """Count trainable and total parameters of the model"""
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Non-trainable parameters: {total_params - trainable_params:,}")
    print(f"Total parameters: {total_params:,}")
save_path = "/kaggle/working/confusion_matrix.png"



def test_model(model, test_loader, device, model_path=MODEL_SAVE_PATH):
    """Evaluate model on test set (prints loss and accuracy, no return)"""
    # Load best model
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    all_preds = []
    all_labels = []
    total_loss = 0.0
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        correct = 0
        total = 0

        for inputs, labels in tqdm(test_loader, desc='Testing'):
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)

            loss = criterion(outputs, labels)
            total_loss += loss.item()

            _, predicted = outputs.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    all_labels = [label + 1 for label in all_labels]
    all_preds = [pred + 1 for pred in all_preds]
    # Compute metrics
    test_loss = total_loss / len(test_loader)
    test_accuracy = 100. * correct / total

    # Print results
    print(f"\nTest Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_accuracy:.2f}%")

    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    cm_percentage = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

    # Plot confusion matrix
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm_percentage,
        annot=True,
        fmt='.2f',
        cmap='Blues',
        xticklabels=list(EMOTION_MAPPING.values()),
        yticklabels=list(EMOTION_MAPPING.values())
    )
    plt.title('Confusion Matrix (%)')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')

    # Save the figure
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"Confusion matrix saved at: {save_path}")

    # Print classification report
    print("\nClassification Report:")
    print(classification_report(
        all_labels,
        all_preds,
        target_names=list(EMOTION_MAPPING.values()),
        digits=4
    ))


def main():
    mobilefacenet = MobileFaceNet(input_size=[112, 112], embedding_size=136).to(DEVICE)
    try:
                face_landback_checkpoint = torch.load(r'/kaggle/working/mobilefacenet_model_best.pth.tar',
                                              map_location=lambda storage, loc: storage)
                mobilefacenet.load_state_dict(face_landback_checkpoint['state_dict'])
                print(f"Tải thành công trọng số từ")
    except Exception as e:
                print(f"Không thể tải trọng số. Lỗi: {e}")
                print("Mô hình sẽ sử dụng trọng số được khởi tạo ngẫu nhiên.")
    
    mobilefacenet.output_layer.linear = nn.Linear(in_features=512, out_features=7, bias=False).to(DEVICE)

    mobilefacenet.output_layer.bn = nn.BatchNorm1d(7).to(DEVICE)
    count_parameters(mobilefacenet)
    model = train_model(
        mobilefacenet,
        train_loader,
        val_loader,
        DEVICE,
        NUM_EPOCHS,
        MODEL_SAVE_PATH
    )
    
    # Test model
    print("\nEvaluating model on test set...")
    test_model(model, test_loader, DEVICE, MODEL_SAVE_PATH)

if __name__ == "__main__":
    main()

Tải thành công trọng số từ
Trainable parameters: 940,942
Non-trainable parameters: 0
Total parameters: 940,942


Epoch 1/120: 100%|██████████| 67/67 [00:12<00:00,  5.28it/s, loss=1.32, acc=59]


Epoch 1/120:
Train Loss: 1.3199, Train Acc: 59.04%
Val Loss: 1.0216, Val Acc: 73.72%
The model which has the best validation accuracy is saved...
Best Epoch 1/120:
Best Validation Accuracy:  73.71615312791783


Epoch 2/120: 100%|██████████| 67/67 [00:11<00:00,  5.70it/s, loss=0.984, acc=75.1]


Epoch 2/120:
Train Loss: 0.9838, Train Acc: 75.06%
Val Loss: 0.9038, Val Acc: 78.34%
The model which has the best validation accuracy is saved...
Best Epoch 2/120:
Best Validation Accuracy:  78.33800186741364


...

Epoch 119/120:
Train Loss: 0.0767, Train Acc: 99.31%
Val Loss: 0.5221, Val Acc: 83.99%


Epoch 120/120: 100%|██████████| 67/67 [00:11<00:00,  5.65it/s, loss=0.0778, acc=99.3]


Epoch 120/120:
Train Loss: 0.0778, Train Acc: 99.32%
Val Loss: 0.5240, Val Acc: 83.57%
Loss plot saved at: /kaggle/working/loss_plot.png
Accuracy plot saved at: /kaggle/working/accuracy_plot.png

Evaluating model on test set...


Testing: 100%|██████████| 21/21 [00:01<00:00, 14.79it/s]



Test Loss: 0.4946
Test Accuracy: 84.90%
Confusion matrix saved at: /kaggle/working/confusion_matrix.png

Classification Report:
              precision    recall  f1-score   support

    surprise     0.8989    0.8218    0.8586       303
        fear     0.6780    0.6349    0.6557        63
     disgust     0.5893    0.4681    0.5217       141
   happiness     0.9164    0.9573    0.9364      1054
     sadness     0.7931    0.8313    0.8118       415
       anger     0.7538    0.7481    0.7510       131
     neutral     0.8274    0.8172    0.8223       569

    accuracy                         0.8490      2676
   macro avg     0.7796    0.7541    0.7654      2676
weighted avg     0.8456    0.8490    0.8465      2676

